# Does AI → top-10% survive on a *long* panel?
### A standalone stress-test of the concentration channel (WID 1980–2024)

In the main framework the escape dividend (F×G on growth) survived every estimator, but
the **concentration channel** — AI's effect on the top-10% income share — was positive
under fixed effects yet **washed out under dynamic GMM and was not IV-identified**. The
diagnosis: the top share is a near-unit-root *stock* (R²≈0.99 on its own lag, ~0.19pp of
movement/yr, 85% of variance cross-country), and **15 years (2010–24) is too short** to
identify a dynamic effect.

This notebook tests the fix: a **long consistent panel back to 1980**. Length helps three
ways — (1) deep lags become valid, strong instruments for the lagged dependent variable,
(2) the Nickell bias shrinks as $O(1/T)$, (3) a **pre-AI baseline** enables a placebo.
The AI regressor still only varies from ~2010, so this cannot manufacture an effect — it
only gives the dynamic estimator enough history to *identify* one if it exists.

> Runs in Colab (needs internet for the WID pull). Load order: run top to bottom.

In [ ]:
# --- setup -------------------------------------------------------------------
import sys, subprocess
def _pip(*p):
    try: subprocess.run([sys.executable,"-m","pip","-q","install",*p],check=True)
    except Exception as e: print("pip skip:",e)
_pip("linearmodels","country_converter","pydynpd","scipy")

import numpy as np, pandas as pd, warnings, io, os, requests
warnings.filterwarnings("ignore")
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS, RandomEffects, FirstDifferenceOLS
from linearmodels.iv import IV2SLS
import country_converter as _cc; cc=_cc.CountryConverter()

# numpy>=2 shims so pydynpd (pre-2.0) runs
def _shim():
    import numpy as _np, math as _m
    if not hasattr(_np,"in1d"): _np.in1d=_np.isin
    try:
        import pydynpd.specification_tests as _st
        _st.float=(lambda x: float(_np.squeeze(x)) if isinstance(x,_np.ndarray) else float(x))
        class _M:
            def __getattr__(s,n): return getattr(_m,n)
            def sqrt(s,x): return _m.sqrt(float(_np.squeeze(x)) if isinstance(x,_np.ndarray) else x)
        _st.math=_M(); return True
    except Exception as e: print("pydynpd unavailable:",e); return False
HAVE_GMM=_shim()
def star(p): return "***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""
def z(s): s=pd.to_numeric(s,errors="coerce"); return (s-s.mean())/s.std()
print("ready")

In [ ]:
# --- CONFIG ------------------------------------------------------------------
CONFIG = {
  # your AI panel from the main framework (has G/ai_pubs, F, H, controls; 2010-2024)
  "AI_PANEL": "unified_panel.csv",        # or merged.csv / final_panel_imputed__5_.csv
  "START_YEAR": 1980,                     # long-panel start
  # long WID top-10 share (consistent DINA methodology, 1820-2024) via OWID:
  "WID_TOP10_URL": "https://ourworldindata.org/grapher/income-share-top-10-before-tax-wid.csv?v=1&csvType=full&useColumnShortNames=false",
  "WID_TOP1_URL" : "https://ourworldindata.org/grapher/income-share-top-1-before-tax-wid.csv?v=1&csvType=full&useColumnShortNames=false",
}
CONTROLS=["log_gdp","trade_pct_gdp","inflation","investment_pct_gdp",
          "pop_growth","credit_to_private_pct_gdp","secondary_enrollment"]
Cl=[f"{c}_l" for c in CONTROLS]

## Upload your files here

Run the next cell and pick **all** the files you want to use at once. You need:
- **your AI panel** — `unified_panel.csv` (or `merged.csv`), and
- *(optional)* the **WID top-share CSVs** downloaded from Our World in Data
  ([top-10%](https://ourworldindata.org/grapher/income-share-top-10-before-tax-wid),
  [top-1%](https://ourworldindata.org/grapher/income-share-top-1-before-tax-wid) →
  each page has a **Download → Full data (CSV)** button).

If you skip the WID files, the notebook will try to fetch them from OWID automatically.
No Google Drive / shared folder needed — everything runs from what you upload.

In [ ]:
# --- upload all needed files (Colab), or use whatever CSVs are already here --
import glob
def upload_files():
    try:
        from google.colab import files
        print("Select ALL files to upload (AI panel + optional WID CSVs)…")
        up=files.upload()                      # writes each file into the working dir
        names=list(up.keys()); print("uploaded:", names); return names
    except Exception:
        names=glob.glob("*.csv"); print("not in Colab — using local CSVs:", names); return names
UPLOADED = upload_files()

def find_csv(must_have=None, name_hints=(), val_range=None):
    """Locate an uploaded/local CSV by required columns, filename hint, or (for WID)
    the magnitude of its value column."""
    cands=glob.glob("*.csv")
    for c in cands:                                    # 1) filename hint
        if any(h in c.lower() for h in name_hints): return c
    for c in cands:                                    # 2) required columns
        try:
            h=pd.read_csv(c,nrows=5)
            if must_have and set(must_have).issubset(h.columns): return c
        except Exception: pass
    if val_range:                                      # 3) OWID share magnitude
        lo,hi=val_range
        for c in cands:
            try:
                h=pd.read_csv(c,nrows=400)
                if {"Code","Year"}.issubset(h.columns):
                    vc=[k for k in h.columns if k not in ("Entity","Code","Year")][-1]
                    v=pd.to_numeric(h[vc],errors="coerce"); m=v.median()
                    m=m/100 if m>1.5 else m
                    if lo<=m<=hi: return c
            except Exception: pass
    return None

In [ ]:
# --- 1. load the LONG, consistent WID top-share series (1980-2024) -----------
def load_wid(name, url, name_hints, val_range):
    """Use an uploaded OWID CSV if present (matched by name or share magnitude),
    else fetch it from OWID. Returns tidy iso3/year/value (fraction)."""
    p=find_csv(name_hints=name_hints, val_range=val_range)
    if p:
        raw=pd.read_csv(p); print(f"  [{name}] using uploaded file: {p}")
    else:
        try:
            raw=pd.read_csv(url, storage_options={"User-Agent":"research/1.0"})
            print(f"  [{name}] fetched from OWID")
        except Exception as e:
            print(f"  [{name}] no upload and fetch failed ({e}) — skipping"); return None
    valcol=[c for c in raw.columns if c not in ("Entity","Code","Year")][-1]
    d=raw.rename(columns={"Code":"iso3","Year":"year",valcol:name})[["iso3","year",name]]
    d=d[d["iso3"].astype(str).str.len()==3].dropna(subset=[name])
    if d[name].median()>1.5: d[name]=d[name]/100.0     # % -> fraction
    return d

top10=load_wid("top10", CONFIG["WID_TOP10_URL"], ("top-10","top10"), (0.25,0.9))
top1 =load_wid("top1",  CONFIG["WID_TOP1_URL"],  ("top-1-","top1"),  (0.03,0.25))
assert top10 is not None, "No top-10 WID data (upload the OWID CSV or enable internet)."
wid=top10.merge(top1,on=["iso3","year"],how="outer") if top1 is not None else top10
wid=wid[wid.year>=CONFIG["START_YEAR"]]
print("WID long panel:", wid.shape, "| years", int(wid.year.min()),"-",int(wid.year.max()),
      "| countries", wid.iso3.nunique())
print("  top10 within-country AR1 (long) =",
      round(wid.sort_values('year').groupby('iso3')['top10'].apply(lambda s:s.autocorr()).mean(),3))

In [ ]:
# --- 2. merge the AI panel (G, F, H, controls) onto the long WID spine -------
AI_PATH = (CONFIG["AI_PANEL"] if os.path.exists(CONFIG["AI_PANEL"])
           else find_csv(must_have=["iso3","year"], name_hints=("unified","merged","panel")))
assert AI_PATH, "No AI panel found — upload unified_panel.csv (or merged.csv) in the upload cell."
print("AI panel:", AI_PATH)
ai=pd.read_csv(AI_PATH, low_memory=False).drop_duplicates(["iso3","year"])
keep=[c for c in ["iso3","year","G","F","F_l","H","ai_pubs_log","G_sci","G_sci_l",
                  "mi_status","income_group"]+CONTROLS+Cl if c in ai.columns]
panel=wid.merge(ai[keep], on=["iso3","year"], how="left").sort_values(["iso3","year"])

# AI activity, year-standardized within the AI era; zero/NA before AI exists
if "ai_pubs_log" in panel: 
    panel["G_fe"]=panel.groupby("year")["ai_pubs_log"].transform(lambda s:(s-s.mean())/s.std())
elif "G" in panel:
    panel["G_fe"]=panel.groupby("year")["G"].transform(lambda s:(s-s.mean())/s.std())
panel["G_fe_l"]=panel.groupby("iso3")["G_fe"].shift(1)
panel["GH_l"]=panel["G_fe_l"]*(1-panel["H"]) if "H" in panel else np.nan
for c in Cl:
    if c not in panel and c[:-2] in panel: panel[c]=panel.groupby("iso3")[c[:-2]].shift(1)

ai_era=panel.dropna(subset=["G_fe_l","top10"])
print(f"long panel rows: {len(panel)} | AI-era estimable rows (G & top10): {len(ai_era)}")
print(f"  countries with BOTH long top10 history AND AI data: "
      f"{ai_era.iso3.nunique()}")
# how much pre-AI history feeds the lagged-DV instruments?
hist=panel.dropna(subset=["top10"]).groupby("iso3")["year"].agg(["min","count"])
print(f"  median first top10 year = {int(hist['min'].median())}, "
      f"median #years top10 per country = {int(hist['count'].median())}")

In [ ]:
# --- 3. static suite on the long panel  (γ₁ = ∂top10/∂G) ---------------------
reg=["G_fe_l","GH_l","F_l"]+[c for c in Cl if c in panel]
d=panel.dropna(subset=["top10"]+reg).set_index(["iso3","year"])
y=d["top10"]; X=sm.add_constant(d[reg])
print("STATIC SUITE (long panel):")
suite={"Pooled":PooledOLS(y,X).fit(cov_type="clustered",cluster_entity=True),
       "Between":BetweenOLS(y,X).fit(),
       "RE":RandomEffects(y,X).fit(cov_type="clustered",cluster_entity=True),
       "FE(2way)":PanelOLS(y,d[reg],entity_effects=True,time_effects=True,
                  drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True),
       "FD":FirstDifferenceOLS(y,d[reg]).fit(cov_type="clustered",cluster_entity=True)}
for n,r in suite.items():
    print(f"  {n:9s} γ₁={r.params['G_fe_l']:+.4f}{star(r.pvalues['G_fe_l'])} N={int(r.nobs)}")

In [ ]:
# --- 4. dynamic GMM with DEEP lags (the whole point of the long panel) -------
# With top10 history back to 1980, lags 2..10 are available as instruments for the
# lagged dependent variable — deep, and (being far in the past) plausibly exogenous.
if HAVE_GMM:
    from pydynpd import regression
    g=panel.copy(); g["id"]=pd.factorize(g["iso3"])[0]+1
    g["top10"]=g["top10"]*100                       # pct points for numerical scale
    # keep the FULL history (incl. pre-AI top10) so deep lags exist; the regression
    # rows are still only where G_fe_l is present, but instruments reach back.
    g=g.dropna(subset=["top10","id","year"]).copy()
    need=["top10","G_fe_l","GH_l","F_l"]
    g2=g.dropna(subset=need).copy()
    print(f"GMM sample: {len(g2)} rows, {g2.id.nunique()} countries "
          f"({int(g2.year.min())}-{int(g2.year.max())}); "
          f"top10 history spans {int(g.year.min())}-{int(g.year.max())}")
    for lbl,tail in [("Difference GMM (deep lags 2:10)",
                      "| gmm(top10, 2:10) gmm(G_fe_l, 2:6) iv(F_l GH_l) | nolevel collapse"),
                     ("System GMM (deep lags 2:10)",
                      "| gmm(top10, 2:10) gmm(G_fe_l, 2:6) iv(F_l GH_l) | collapse")]:
        print("\n"+"="*58+"\n"+lbl+"\n"+"="*58)
        try: regression.abond(f"top10 L1.top10 G_fe_l GH_l F_l {tail}", g2, ["id","year"])
        except Exception as e: print("  GMM failed:",e)
    print("\nCheck: AR(2) p>0.10, Hansen p>0.10, and — critically — whether G_fe_l is now")
    print("SIGNIFICANT with the deep-lag instruments that only the long panel provides.")
else:
    print("pydynpd unavailable.")

In [ ]:
# --- 5. PRE-AI PLACEBO: the test only a long panel can run -------------------
# If AI causes concentration, a FAKE take-off placed BEFORE AI existed should do
# nothing. We give each country a random fake take-off in its own pre-2010 window
# (so it varies within-year → not absorbed by year effects) and repeat many times;
# the placebo coefficient should average ~0 and be significant only ~5% of the time.
pre=panel[panel.year<2010].dropna(subset=["top10"]).copy()
n_pre, n_ctry = len(pre), pre.iso3.nunique()
print(f"pre-2010 top10 rows: {n_pre} across {n_ctry} countries")
if n_pre>300 and n_ctry>15:
    rng=np.random.default_rng(0)
    span=pre.groupby("iso3")["year"].agg(["min","max"])
    elig=span[(span["max"]-span["min"])>=5].index                 # ≥6 yrs of pre-2010 history
    base=pre[pre.iso3.isin(elig)].copy()
    coefs=[]; sig=0; B=100
    for _ in range(B):
        fake={c: rng.integers(int(span.loc[c,"min"])+2, int(span.loc[c,"max"])) for c in elig}
        base["placebo"]=(base["year"]>=base["iso3"].map(fake)).astype(float)
        dd=base.dropna(subset=["placebo"]).set_index(["iso3","year"])
        try:
            rr=PanelOLS.from_formula("top10 ~ 1 + placebo + EntityEffects + TimeEffects",
                     data=dd,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
            if "placebo" in rr.params.index:
                coefs.append(rr.params["placebo"]); sig += rr.pvalues["placebo"]<0.05
        except Exception: pass
    coefs=np.array(coefs)
    if len(coefs):
        print(f"Placebo over {len(coefs)} random pre-AI take-offs:")
        print(f"  mean placebo coef = {coefs.mean():+.4f}  (should be ≈ 0)")
        print(f"  significant at 5% = {100*sig/len(coefs):.0f}% of draws  (should be ≈ 5% under the null)")
        print("  ≈0 mean and ≈5% hit-rate ⇒ no spurious pre-trend the real result could ride on;")
        print("  a high hit-rate ⇒ top shares trend in ways that would inflate any post-2010 effect.")
else:
    print("  → placebo skipped: the long WID history is NOT loaded (little/no pre-2010 top10).")
    print("    Download the OWID top-10 CSV (Download → Full data) from")
    print("    ourworldindata.org/grapher/income-share-top-10-before-tax-wid and upload it, then re-run.")

### 5b — Which countries are driving the placebo failures?

If the placebo's hit-rate is well above ~5%, some countries' pre-2010 top-share history
has real discontinuities (crises, tax reforms, transitions, or noisy pre-1990 WID
modelling) that a random "fake AI" shock can pick up. This cell scores each country by
how large its biggest unexplained pre-2010 jump is, flags the most volatile decile, and
re-runs both the placebo and the headline FE **excluding** them — so you can see whether
a handful of countries are contaminating the whole panel or whether the instability is
pervasive.

In [ ]:
# --- 5b. flag break-driving countries; re-run placebo + FE without them -----
def _break_score(s):
    """Largest jump in a country's DETRENDED pre-2010 top10 series — big values mean
    a real discontinuity a random placebo year could easily land on and 'detect'."""
    s=s.sort_values("year")
    if len(s)<6: return np.nan
    t=np.arange(len(s)); y=s["top10"].values
    coef=np.polyfit(t,y,1); resid=y-np.polyval(coef,t)
    d=np.diff(resid)
    return np.max(np.abs(d)) if len(d) else np.nan

pre=panel[panel.year<2010].dropna(subset=["top10"]).copy()
if len(pre)>300 and pre.iso3.nunique()>15:
    scores=pre.groupby("iso3").apply(_break_score).dropna().sort_values(ascending=False)
    cutoff=scores.quantile(0.90)
    flagged=scores[scores>=cutoff].index.tolist()
    print(f"Flagged {len(flagged)} countries (top decile of pre-2010 break volatility):")
    print(" ", flagged)
    print("\n  (largest 5 scores)"); print(scores.head(5).round(4).to_string())

    def _run_placebo(frame,B=100,seed=1):
        rng2=np.random.default_rng(seed)
        span=frame.groupby("iso3")["year"].agg(["min","max"])
        elig=span[(span["max"]-span["min"])>=5].index
        base=frame[frame.iso3.isin(elig)].copy()
        coefs=[]; sig=0
        for _ in range(B):
            fake={c: rng2.integers(int(span.loc[c,"min"])+2, int(span.loc[c,"max"])) for c in elig}
            base["placebo"]=(base["year"]>=base["iso3"].map(fake)).astype(float)
            dd=base.dropna(subset=["placebo"]).set_index(["iso3","year"])
            try:
                rr=PanelOLS.from_formula("top10 ~ 1 + placebo + EntityEffects + TimeEffects",
                         data=dd,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
                if "placebo" in rr.params.index:
                    coefs.append(rr.params["placebo"]); sig+=rr.pvalues["placebo"]<0.05
            except Exception: pass
        return np.array(coefs), sig

    c0,s0=_run_placebo(pre)
    c1,s1=_run_placebo(pre[~pre.iso3.isin(flagged)])
    print(f"\nPlacebo hit-rate  FULL sample      : {100*s0/max(len(c0),1):.0f}%  (mean coef {c0.mean():+.4f})")
    print(f"Placebo hit-rate  EXCLUDING flagged: {100*s1/max(len(c1),1):.0f}%  (mean coef {c1.mean():+.4f})")

    reg=["G_fe_l","GH_l","F_l"]+[c for c in Cl if c in panel]
    d_all=panel.dropna(subset=["top10"]+reg).set_index(["iso3","year"])
    d_cln=panel[~panel.iso3.isin(flagged)].dropna(subset=["top10"]+reg).set_index(["iso3","year"])
    r_all=PanelOLS.from_formula("top10 ~ 1 + "+" + ".join(reg)+" + EntityEffects + TimeEffects",
                data=d_all,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
    r_cln=PanelOLS.from_formula("top10 ~ 1 + "+" + ".join(reg)+" + EntityEffects + TimeEffects",
                data=d_cln,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
    print(f"\nFE γ₁  FULL sample      : {r_all.params['G_fe_l']:+.4f}{star(r_all.pvalues['G_fe_l'])}  N={int(r_all.nobs)}")
    print(f"FE γ₁  EXCLUDING flagged: {r_cln.params['G_fe_l']:+.4f}{star(r_cln.pvalues['G_fe_l'])}  N={int(r_cln.nobs)}")

    print("\nRead:")
    print("  • hit-rate drops toward ~5% after exclusion  ⇒ a handful of volatile countries were")
    print("    contaminating the whole panel; the FE estimate on the clean subset is more credible.")
    print("  • hit-rate STAYS high after exclusion         ⇒ instability is pervasive (e.g. widespread")
    print("    noise in modelled pre-1990 WID estimates), not a few outliers — treat the entire FE/GMM")
    print("    battery on this long panel with real caution, and lean on the IV result instead.")
    print("  • if γ₁ changes sign or loses significance after exclusion, the flagged countries were")
    print("    doing a lot of the work in the full-sample estimate.")
else:
    print("Not enough pre-2010 coverage to run the diagnostic — upload the long WID history first.")

In [ ]:
# --- 6. IV on the long panel + verdict --------------------------------------
verdict=[]
if {"G_sci_l"}.issubset(panel.columns):
    panel["GHc_l"]=panel["G"]*(1-panel["H"]) if {"G","H"}.issubset(panel.columns) else np.nan
    en=["G"]; iv_=["G_sci_l"]; ex=["F_l"]+[c for c in Cl if c in panel]
    di=panel.dropna(subset=["top10"]+en+iv_+ex+["iso3","year"]).copy()
    if len(di)>200:
        for c in ["top10"]+en+iv_+ex:
            di[c]=(di[c]-di.groupby("iso3")[c].transform("mean")
                        -di.groupby("year")[c].transform("mean")+di[c].mean())
        di["const"]=1
        ivc=IV2SLS(di["top10"],di[["const"]+ex],di[en],di[iv_]).fit(cov_type="clustered",clusters=di["iso3"])
        F=ivc.first_stage.diagnostics.loc["G","f.stat"]
        print(f"IV (Scimago→AI): γ₁={ivc.params['G']:+.4f}{star(ivc.pvalues['G'])} | first-stage F={F:.1f}"
              +(" (usable)" if F>10 else " (weak)"))

print("\n" + "="*60)
print("VERDICT — did the long panel rescue the concentration channel?")
print("="*60)
print("Compare against the SHORT-panel result (FE +0.074***, GMM insignificant, IV weak):")
print("  • If GMM G_fe_l is now significant with AR(2) & Hansen passing → the effect")
print("    survives once enough history identifies the dynamics (length was the problem).")
print("  • If it is STILL insignificant with deep lags and a clean placebo → the FE result")
print("    is a genuine within-country association without a dynamically-identified causal")
print("    effect (the honest conclusion the main notebook reached).")
print("  • Either way, the pre-AI placebo + long baseline make the claim far more credible")
print("    than the 15-year window could.")

In [ ]:
# --- 7. save the long panel --------------------------------------------------
out="long_panel_concentration.csv"
panel.to_csv(out,index=False); print("wrote",out, panel.shape)
try:
    from google.colab import files; files.download(out)
except Exception: pass

### How to read this

- The **AI coefficient is still only identified from ~2010 onward** — the long history
  cannot invent an AI effect. What it *can* do is give the dynamic estimator valid, deep
  instruments and a pre-AI baseline, so that if a within-country causal effect exists, the
  GMM now has the power to see it.
- **Best case:** `G_fe_l` turns significant in system GMM with AR(2) and Hansen passing —
  length was the binding constraint, and the concentration drag is real and dynamic.
- **Honest case:** it stays insignificant even with deep lags while the pre-AI placebo is
  null — then the fixed-effects association is genuine but not dynamically identified, and
  should be reported as a correlation, exactly as the main framework concluded.
- **Contaminated case:** the placebo hit-rate is well above 5% — then check 5b. If
  excluding the flagged countries brings the hit-rate down and changes γ₁, the full-sample
  FE/GMM results were partly riding on a handful of countries' unrelated historical breaks,
  and the **IV estimate (immune to this problem) is the more trustworthy number** to report.
- **Caveat:** WID top shares before ~1990 are modelled/extrapolated for many countries;
  the deep lags are noisier than recent data. Treat pre-1990 instruments with due caution.